In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

True

In [2]:
!pip install pypdf

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader

# pypdf 설치 확인 

In [7]:
# pypdf 설치 확인
try:
    import pypdf
    print(f"pypdf: {pypdf.__version__}")
except ImportError:
    print("pypdf 미설치 → pip install pypdf")

pypdf: 6.9.2


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [6]:
loader = PyPDFLoader("data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf")
pages = loader.load()

print(f"총 페이지 수: {len(pages)}")


총 페이지 수: 93


In [8]:
from dotenv import load_dotenv
import os
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

# 기존 인덱스 없으면 생성
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함")

# 상태 확인
f_index = pc.Index(INDEX_NAME)
print(f_index .describe_index_stats())

finance-bok 생성 완료


c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


# PDF 파일 로드

In [10]:
PDF_PATH = "data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 300자):\n{docs[0].page_content[:300]}")
print(f"\n메타데이터: {docs[0].metadata}")


총 페이지 수: 93

첫 페이지 내용 (앞 300자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


# Splitter 생성

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

# Splitter로 쪼개기

In [12]:
splits = splitter.split_documents(docs) #원본 문서의 메타 정보를 가져와 설정

In [13]:
print(f"총 청크 수: {len(splits)}")
print(f"\n첫 번째 청크:\n{splits[0].page_content}")
print(f"\n메타데이터: {splits[0].metadata}")

총 청크 수: 249

첫 번째 청크:
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


# embedding model 생성

In [14]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 업서트 

In [15]:
from langchain_pinecone import PineconeVectorStore
# 업서트
vectorstore = PineconeVectorStore.from_documents(
    documents=splits,
    embedding=embedding_model ,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
print(f"업서트 완료 — 총 {len(splits)}개 청크")

업서트 완료 — 총 249개 청크


# 업서트 확인

In [17]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'total_vector_count': 249,
 'vector_type': 'dense'}


In [ ]:
# retriever

In [20]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [24]:
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": NAMESPACE}
)

# querry 준비

In [25]:
# 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "고용 시장 전망은?"
]

In [26]:
for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.6.0) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에

[ 질문 ] 물가 상승률 전망은?
  1. (p.53.0) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 
  2. (p.53.0) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련
한 불확실성이 크며, 
  3. (p.11.0) < 요약 6/8 > 
6  
  
전망의 리스크 
 
 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련한 불확
실성이 크며, 물가의 경우 유가, 환

[ 질문 ] 고용 시장 전망은?
  1. (p.10.0) ▪ 상품수지는 반도체가격의 큰 폭 상승 등으로 흑자규모가 크게 늘어날 전망이다. 
서비스수지는 경기회복에 따른 산업서비스특허사용료 등 수요 증가, 디지털서비스플랫폼 구
독료 등 지
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.53.0) 40 


# RAG CHAIN 구성

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 프롬프트

In [29]:
prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

# RAG CHAIN

In [30]:
rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

# RAG TEST

In [31]:
question = "수출 전망은 어떻게 되나요?"
rag_chain.invoke(question)

'수출은 견조한 흐름을 이어갈 것으로 전망됩니다.'

In [32]:
# RAG 테스트
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ A ] 소비자물가 상승률은 2.0%로 전망됩니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 수출은 견조한 흐름을 이어갈 것으로 전망됩니다. 2024년 수출은 6,836억 달러에서 시작하여 2025년에는 7,093억 달러, 2026년에는 7,952억 달러로 증가할 것으로 예상됩니다.

